# TTZ YOLOv8n v3 Fine-tune
## 推筒子水平牌偵測訓練

### 使用方法
執行表：Runtime → Run all
訓練完會自動匯出 ONNX，下載 `ttz_v3_160.onnx` 即可部署到 Pi

In [ ]:
# @title 下載 dataset
import os, zipfile, glob, urllib.request

DST = '/content/mahjong_v3_dataset'
if not os.path.exists(DST):
    url = 'https://github.com/Powerck7788/ttz_v3_dataset/releases/download/v1.0/mahjong_v3_dataset.zip'
    zip_path = '/content/mahjong_v3_dataset.zip'
    print('Downloading dataset (546MB)...')
    urllib.request.urlretrieve(url, zip_path)
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/')
    os.remove(zip_path)
    print('Done')
else:
    print('Already exists')

In [ ]:
# @title 驗證 dataset
train_imgs = glob.glob(f'{DST}/images/train/*.jpg')
val_imgs = glob.glob(f'{DST}/images/val/*.jpg')
print(f'Train: {len(train_imgs)} imgs')
print(f'Val:   {len(val_imgs)} imgs')
!cat {DST}/data.yaml

In [ ]:
# @title 安裝 ultralytics
!pip install -q ultralytics
import torch
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# @title 開始訓練（約 30-40 分鐘）
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{DST}/data.yaml',
    epochs=200,
    patience=30,
    batch=32,
    imgsz=640,
    device='0',
    project='/content',
    name='yolov8n_v3',
    optimizer='AdamW',
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=3,
    degrees=5,
    translate=0.1,
    scale=0.5,
    shear=2,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.5,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    val=True,
    plots=True,
    save=True,
    save_period=10,
    amp=True,
)
print('Training complete!')

In [ ]:
# @title 查看結果
import pandas as pd
csv = '/content/yolov8n_v3/results.csv'
if os.path.exists(csv):
    df = pd.read_csv(csv)
    best = df.iloc[df['metrics/mAP50(B)'].idxmax()]
    print(f"Best epoch {int(best['epoch'])}:")
    print(f"  mAP50:    {best['metrics/mAP50(B)']:.4f}")
    print(f"  mAP50-95: {best['metrics/mAP50-95(B)']:.4f}")
    print(f"  P: {best['metrics/precision(B)']:.4f}")
    print(f"  R: {best['metrics/recall(B)']:.4f}")

In [ ]:
# @title 匯出 ONNX + 下載
from ultralytics import YOLO
model = YOLO('/content/yolov8n_v3/weights/best.pt')
model.export(format='onnx', imgsz=160, half=True, simplify=True)

import shutil
shutil.copy('/content/yolov8n_v3/weights/best_160.onnx', '/content/ttz_v3_160.onnx')
shutil.copy('/content/yolov8n_v3/weights/best.pt', '/content/ttz_v3.pt')

!ls -lh /content/ttz_v3_160.onnx
print('\nDownloading ONNX...')
from google.colab import files
files.download('/content/ttz_v3_160.onnx')